# Fine-tuning the Yorùbá vowel encoder

Everything in this project so far has trained the **classifier** on top of a
**frozen** encoder. `LyngualLabs/mms-300m-yoruba-english` has never had its
300M weights touched here. That is why the numbers moved the way they did:

| change | accuracy | macro F1 |
|---|---|---|
| 39 hand-designed features | 0.595 | 0.432 |
| **swap to the frozen encoder** | **0.873** | **0.811** |
| retrain aligner, 4.2x the vowels | 0.895 | 0.834 |

Replacing the representation was worth +0.379 macro F1. Quadrupling the data on
top of it was worth +0.023. The encoder is the ceiling.

This notebook unfreezes it and trains it on Yorùbá vowel identity directly.

**What it needs**
- GPU (T4 or P100, 16 GB) — Settings → Accelerator
- The `finetune_pack` dataset attached (built by `scripts/build_finetune_pack.py`)
- Internet on, to pull the encoder from the GitHub release

**What it produces** — a `model.safetensors` to drop into `models/lyngual-yoruba/`,
after which you re-extract features and retrain the classifier locally.

In [ ]:
!pip -q install transformers==4.44.2 'huggingface_hub<1.0' safetensors

import torch, os, sys
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{name}, {total:.1f} GB")
else:
    raise SystemExit("Enable a GPU accelerator before running this notebook.")

## 1. The encoder

Pulled from HuggingFace, not from the project's GitHub release. The release is
in a **private** repo, and GitHub answers unauthenticated requests for private
release assets with a 404 rather than a 401 — so the download fails in a way
that looks like a missing file. `scripts/fetch_models.py` works around that with
the `gh` CLI and a token, which is not worth arranging inside a Kaggle session.

It is the same checkpoint either way: the weights the engine ships are the
unmodified upstream ones (sha256 `766a4e6c…`), never altered by this project.

Kaggle needs **Internet enabled** (Settings → Internet) for this cell.

In [ ]:
from pathlib import Path
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

# The public upstream checkpoint. If you later fine-tune and want to continue
# from your own result, upload it as a Kaggle dataset and point ENCODER at it
# instead -- a private GitHub release cannot be reached from here without a
# token.
ENCODER = "LyngualLabs/mms-300m-yoruba-english"

probe = Wav2Vec2Model.from_pretrained(ENCODER)
print(f"{sum(p.numel() for p in probe.parameters())/1e6:.0f}M parameters, "
      f"{len(probe.encoder.layers)} layers, hidden {probe.config.hidden_size}")
del probe

## 2. The data

`build_finetune_pack.py` writes fixed 0.4 s windows with the vowel centred, plus
a manifest recording where inside each window the vowel actually sits — so
training pools over the vowel rather than the padding, matching what the engine
does at inference.

Every nasal in the corpus is present; the common classes are capped. `ẹn` is
0.53% of Yorùbá vowels, so a uniform sample would have spent its budget on
classes that are already solved.

In [ ]:
import numpy as np, pandas as pd

PACK = Path("/kaggle/input/yoruba-finetune-pack")
if not PACK.exists():
    candidates = sorted(Path("/kaggle/input").glob("*/manifest.tsv"))
    if not candidates:
        raise SystemExit("Attach the finetune_pack dataset (Add Input).")
    PACK = candidates[0].parent
print("pack:", PACK)

manifest = pd.read_csv(PACK / "manifest.tsv", sep="\t")
print(f"{len(manifest):,} clips")
print(manifest.label.value_counts().to_string())
print("\nby source:"); print(manifest.source.value_counts().to_string())

### Speaker-disjoint split

Split by speaker, never by clip — the same speaker on both sides inflates every
number. The hash is the one the local trainer uses, so the split is reproducible
and comparable.

The held-out side deliberately spans SLR86, IroyinSpeech, Common Voice **and**
YECS. Restricting it to studio corpora left `ẹn` with 26 examples, which meant
the class could be trained but never scored.

In [ ]:
import hashlib

def held_out(speaker, percent=20):
    return int(hashlib.md5(speaker.encode()).hexdigest()[:8], 16) % 100 < percent

manifest["hold"] = manifest.speaker.map(held_out)
train_rows = manifest[~manifest.hold].reset_index(drop=True)
valid_rows = manifest[manifest.hold].reset_index(drop=True)
print(f"train {len(train_rows):,} clips / {train_rows.speaker.nunique():,} speakers")
print(f"valid {len(valid_rows):,} clips / {valid_rows.speaker.nunique():,} speakers")
print("\nheld-out support per class:")
print(valid_rows.label.value_counts().to_string())

In [ ]:
from torch.utils.data import Dataset, DataLoader

LABELS = sorted(manifest.label.unique())
INDEX = {name: i for i, name in enumerate(LABELS)}
print(len(LABELS), "classes:", LABELS)

class Vowels(Dataset):
    """Clips plus the frame range the vowel occupies inside each window."""

    def __init__(self, rows, pack):
        self.rows = rows
        # Shards are memory-mapped: the pack is several GB and only the rows
        # actually drawn need to be resident.
        self.shards = {s: np.load(pack / f"clips_{s:03d}.npy", mmap_mode="r")
                       for s in sorted(rows.shard.unique())}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows.iloc[i]
        clip = np.asarray(self.shards[r.shard][r.row], dtype=np.float32) / 32768.0
        return (torch.from_numpy(clip),
                torch.tensor(INDEX[r.label]),
                torch.tensor([r.vowel_start, r.vowel_end]))

train_set, valid_set = Vowels(train_rows, PACK), Vowels(valid_rows, PACK)
BATCH = 24
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_set, batch_size=BATCH, shuffle=False, num_workers=2)
print(f"{len(train_loader):,} train batches")

## 3. The model

The encoder is **truncated to 16 layers**, because layer 16 is what the engine
reads at inference. Layers 17–24 exist only to serve the CTC head we are not
using, so keeping them would cost a third of the compute and train weights
nothing consumes.

The convolutional feature extractor stays frozen — standard for wav2vec2
fine-tuning, and it is the part least in need of adjustment.

Pooling is a mean over the vowel's own frames, at one frame per 20 ms, matching
`Wav2VecExtractor.measure_segment` exactly. Training on a different pooling than
inference uses is a silent way to lose everything gained.

In [ ]:
from transformers import Wav2Vec2Model
import torch.nn as nn

FRAME_S, LAYER = 0.02, 16

class VowelEncoder(nn.Module):
    def __init__(self, path, classes):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained(path)
        self.encoder.encoder.layers = self.encoder.encoder.layers[:LAYER]
        self.encoder.freeze_feature_encoder()
        self.head = nn.Linear(self.encoder.config.hidden_size, classes)

    def forward(self, audio, spans):
        frames = self.encoder(audio).last_hidden_state      # (B, T, 1024)
        pooled = []
        for i, (start, end) in enumerate(spans):
            a = int(start) // int(FRAME_S * 16000)
            b = max(int(end) // int(FRAME_S * 16000), a + 1)
            b = min(b, frames.shape[1])
            a = min(a, b - 1)
            pooled.append(frames[i, a:b].mean(dim=0))
        return self.head(torch.stack(pooled))

model = VowelEncoder(ENCODER, len(LABELS)).cuda()
model.encoder.gradient_checkpointing_enable()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{trainable/1e6:.0f}M trainable parameters")

### Optimiser

Two learning rates. The encoder already knows a great deal about speech and a
large step would destroy it, so it moves at 1e-5 while the freshly initialised
head moves at 1e-3.

The loss is class-weighted. Even after exhausting the rare classes, `ẹn` is a
fraction of `i`, and an unweighted loss would simply learn to ignore it.

In [ ]:
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight

counts = train_rows.label.map(INDEX).values
weights = compute_class_weight("balanced", classes=np.arange(len(LABELS)), y=counts)
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(weights, dtype=torch.float32).cuda())
print({LABELS[i]: round(float(w), 2) for i, w in enumerate(weights)})

optimiser = AdamW([
    {"params": model.encoder.parameters(), "lr": 1e-5},
    {"params": model.head.parameters(),    "lr": 1e-3},
], weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")

## 4. Train

Kaggle sessions stop at 12 hours, so a checkpoint is written after every epoch
and the best macro F1 kept. One epoch is roughly an hour; two or three is
usually where this stops improving.

Watch **macro F1 and the `ẹn` column**, not accuracy. Accuracy is dominated by
`i` and `a`, which were never the problem.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import time

def evaluate():
    model.eval()
    truth, predicted = [], []
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        for audio, y, spans in valid_loader:
            out = model(audio.cuda(non_blocking=True), spans)
            predicted += out.argmax(dim=1).cpu().tolist()
            truth += y.tolist()
    macro = f1_score(truth, predicted, average="macro", zero_division=0)
    per = f1_score(truth, predicted, average=None,
                   labels=list(range(len(LABELS))), zero_division=0)
    return accuracy_score(truth, predicted), macro, per

EPOCHS = 3
best = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train(); started = time.time(); running = 0.0
    for step, (audio, y, spans) in enumerate(train_loader, start=1):
        optimiser.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            loss = criterion(model(audio.cuda(non_blocking=True), spans), y.cuda())
        scaler.scale(loss).backward()
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimiser); scaler.update()
        running += loss.item()
        if step % 200 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} "
                  f"loss {running/step:.4f} "
                  f"{(time.time()-started)/60:.0f} min", flush=True)

    accuracy, macro, per = evaluate()
    print(f"\nepoch {epoch}: accuracy {accuracy:.4f}  macro F1 {macro:.4f}")
    print("  " + "  ".join(f"{n}={per[i]:.3f}" for i, n in enumerate(LABELS)))
    if macro > best:
        best = macro
        model.encoder.save_pretrained("/kaggle/working/finetuned")
        torch.save(model.head.state_dict(), "/kaggle/working/finetuned/head.pt")
        print(f"  saved (best macro F1 {best:.4f})")

## 5. Taking it home

`save_pretrained` writes a truncated 16-layer encoder. That is deliberate and
matches what the engine reads, but it means the result is **not** a drop-in
replacement for the 24-layer checkpoint unless `Wav2VecExtractor` is told the
model now ends at layer 16.

Two ways to use it:

1. **Set `DEFAULT_LAYER = 16` and point at the fine-tuned directory** — the
   extractor already takes `hidden_states[16]`, which is the final layer here.
2. Merge the fine-tuned layers back into the full 24-layer checkpoint if you
   want to keep the original shape.

Either way the pipeline afterwards is: re-extract features, retrain the
classifier, recalibrate. The classifier is fit to the *old* representation and
will be wrong for the new one.

```bash
# locally, after downloading /kaggle/working/finetuned
bash scripts/extract_features_sliced.sh          # re-extract
python scripts/train_vowel_v2.py models/w2v_slices/*.npz
python scripts/calibrate_reliability.py --aligned output/slr86_new
```

Compare against the numbers this notebook started from — 0.895 accuracy,
**0.834 macro F1**, and whatever `ẹn` scores on the widened held-out set. If
macro F1 has not moved, the fine-tune did not earn its compute, and that is a
real result worth recording rather than repeating.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/finetuned_encoder", "zip",
                    "/kaggle/working/finetuned")
print("download /kaggle/working/finetuned_encoder.zip")
print(f"best macro F1 this run: {best:.4f}")